In [5]:
### Imports

# I am importing the main libraries I need for working with the dataset,
# cleaning the text, training the models, and checking the results.

import pandas as pd
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [6]:
### Load the dataset

# I already uploaded the spam.csv file, so I am loading it into a pandas DataFrame.
# This dataset has two main columns: label and message.

df = pd.read_csv("spam.csv")

df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
### Check the dataset

# I am checking the shape to see how many rows and columns are in the dataset.
# I am also checking the column names to make sure everything loaded correctly.

print("Dataset shape:", df.shape)
print("Columns:", df.columns)

Dataset shape: (5572, 2)
Columns: Index(['v1', 'v2'], dtype='str')


In [8]:
### Check for missing values

# Before training the model, I want to make sure there are no missing labels or messages.
# Missing values could cause errors later when cleaning or vectorizing the text.

df.isnull().sum()   

v1    0
v2    0
dtype: int64

In [ ]:
### Check spam and ham distribution

# This shows how many messages are ham and how many are spam.
# It is useful because spam datasets are often imbalanced, meaning there are usually more ham messages.

df = df[["v1", "v2"]]
df.columns = ["label", "message"]

df.head()

KeyError: 'label'

In [ ]:
### Clean the text

# This function cleans each message before it goes into the machine learning model.
# I am making the text lowercase, removing numbers, removing punctuation,
# and fixing extra spaces.

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)

df.head()

In [ ]:
### Convert labels into numbers

# Machine learning models need numbers instead of words.
# I am changing ham to 0 and spam to 1.

df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

df.head()

In [ ]:
### Split the data

# I am separating the cleaned messages from the labels.
# Then I split the data into training and testing sets.
# The model learns from the training set and is tested on messages it has not seen before.

X = df["clean_message"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

In [ ]:
# I am also removing common English stop words like "the", "is", and "and".
# max_features keeps the model focused on the most useful words.

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
#train the model
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_predictions = nb_model.predict(X_test_tfidf)

In [ ]:
### Evaluate the Naive Bayes model

# I am checking accuracy, precision, recall, F1-score, and the confusion matrix.
# These scores help show how well the model is separating spam from ham.

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_predictions))

print("\nClassification Report:")
print(classification_report(y_test, nb_predictions, target_names=["ham", "spam"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, nb_predictions))

In [ ]:
# I am using Logistic Regression as the second model.
# This gives me another classifier to compare with Naive Bayes.

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

lr_predictions = lr_model.predict(X_test_tfidf)

In [ ]:
### Evaluate the Logistic Regression model

# I am using the same evaluation metrics so both models can be compared fairly.

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_predictions))

print("\nClassification Report:")
print(classification_report(y_test, lr_predictions, target_names=["ham", "spam"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))

In [ ]:
### Compare the models

# This table makes it easier to compare the two models side by side.
# Accuracy is helpful, but I should also look at recall and F1-score for spam.

results = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_test, nb_predictions),
        accuracy_score(y_test, lr_predictions)
    ]
})

results

In [ ]:
### Test the model with new messages

# I am testing the trained Logistic Regression model with my own example messages.
# This helps show how the model works on new text.

custom_messages = [
    "Congratulations you won a free prize click now",
    "Hey are you coming to class tomorrow",
    "URGENT your account has been selected for a cash reward",
    "Can you send me the homework notes"
]

custom_cleaned = [clean_text(message) for message in custom_messages]
custom_tfidf = tfidf.transform(custom_cleaned)

custom_predictions = lr_model.predict(custom_tfidf)

for message, prediction in zip(custom_messages, custom_predictions):
    label = "spam" if prediction == 1 else "ham"
    print(message, "-->", label)

NameError: name 'clean_text' is not defined